# Bug 9 PoC - DOM XSS in warninglabelrenderer.js (Lines 119, 140)

**Type:** Stored DOM XSS (no code execution required)  
**Severity:** High  
**File:** `profiler-plugin/frontend/lib/warninglabelrenderer.js`  

## Vulnerability

The profiler plugin's `contentChangedHook()` fires on `cellModel.contentChanged`,
which triggers on **any cell edit** — including simply typing in the cell source.
It does NOT require cell execution.

When triggered, the hook reads `output.data['text/html']` from the cell's existing
outputs. If the HTML matches `hasYarnTable()` (contains `'YARN'` and `'.emr-proxy-link'`),
it calls `saveYarnTable()` which passes the raw HTML to jQuery:

```js
const htmlNode = $(output.data['text/html']);  // line 119 — jQuery parses raw HTML into DOM
```

jQuery instantiates DOM nodes immediately. An `<img src=x>` node fires its `onerror`
handler the moment jQuery creates it, before it's ever inserted into the page.
This bypasses JupyterLab's notebook trust/sanitization entirely.

## How to Trigger

1. Ensure the **profiler plugin** (`emr-profilers-plugin`) is installed and active
2. Open this notebook — the cells below have **pre-populated `text/html` outputs**
   containing a YARN table with an XSS payload
3. **Click into any code cell below and type a single character** (e.g. press Space)
4. `contentChanged` fires → `contentChangedHook` reads the existing outputs →
   `saveYarnTable()` jQuery-parses the HTML → `alert()` fires

**No cell execution. No kernel needed. No notebook trust bypass needed.**

## Attack Scenario

An attacker shares a `.ipynb` file (email, S3, git, JupyterHub shared directory)
with crafted `text/html` outputs embedded in the notebook JSON. The victim opens
the notebook and begins editing — XSS fires on the first keystroke in any cell
that carries the malicious output.

In [1]:
# >>> CLICK INTO THIS CELL AND PRESS ANY KEY <<<
#
# This cell has a pre-populated text/html output (embedded in the .ipynb JSON)
# that looks like a normal YARN application table.
#
# Typing here fires cellModel.contentChanged (source text changed),
# which triggers contentChangedHook → hasYarnTable() matches →
# saveYarnTable() calls $(output.data['text/html']) at line 119 →
# jQuery parses <img src=x onerror=...> into a DOM node → XSS fires.
#
# The victim never executes this cell. The payload is in the OUTPUT, not the source.

YARN Application ID,Kind,State,Spark UI,Driver log
application_1234567890123_0001,spark,idle,Spark UI,Link


In [2]:
# Variant 2: XSS payload INSIDE the .emr-proxy-link element.
#
# This targets line 140 as well: after jQuery parses at line 119,
# saveYarnTable() writes the HTML back via:
#   output.setData({ data: { 'text/html': ['<table>' + htmlNode.html() + '</table>'] } })
#
# The malicious HTML persists in the output, so it re-triggers on
# every subsequent contentChanged event (every keystroke).

YARN Application ID,Kind,State,Spark UI,Driver log
application_9876543210987_0002,pyspark,idle,Spark UI,Link


## Execution Flow

```
Victim types in cell
  → cellModel.contentChanged signal fires
  → contentChangedHook(widget, model) [line 92]
  → iterates codeCellModel.outputs [line 95]
  → reads output.data['text/html'] [line 97]
  → hasYarnTable(data) returns true [line 98]
      (data contains 'YARN' and '.emr-proxy-link')
  → saveYarnTable(codeCellModel, output) [line 99]
  → $(output.data['text/html'])  [line 119]  ← XSS: jQuery parses raw HTML
  → htmlNode.html() written back [line 140]  ← payload persists in output
```

## Why This Bypasses Notebook Trust

JupyterLab's trust model sanitizes `text/html` outputs when **rendering** them
in the notebook UI. However, the profiler plugin reads the raw output data
directly from the cell model and passes it to jQuery — completely bypassing
the sanitization pipeline. The HTML is never rendered through JupyterLab's
output renderer; it's parsed in a side channel by the plugin.